In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("../data/headlines_seen_train.parquet")

In [3]:
## Show the first three words for each headline
df["first_three_words"] = df["headline"].apply(lambda x: " ".join(x.split()[:3]))


In [4]:
remove_verbs = lambda x: " ".join([word for word in x.split() if not word.endswith("ing") or word.istitle()])
# The word "Calvis" ends with "s" but should not be removed, so we will remove words that end with "s" but is not capitalized
remove_s_words = lambda x: " ".join([word for word in x.split() if not word.endswith("s") or word.istitle()])
df["first_three_words_no_verbs"] = df["first_three_words"].apply(remove_verbs)
df["first_three_words_no_verbs_no_s"] = df["first_three_words_no_verbs"].apply(remove_s_words)

In [5]:
df["first_three_words_no_verbs_no_s"].value_counts()

first_three_words_no_verbs_no_s
Zelval Energy              161
Calvos Genomics            159
Crevol Retail Group        154
Frelol Software            141
Myrnep Technologies        141
                          ... 
Urvel Grid CEO               1
Zrovex Industries Chief      1
Halven Investments CEO       1
Volval Devices CEO           1
Zelvon Biosciences CTO       1
Name: count, Length: 335, dtype: int64

In [6]:
## Remove words like CEO, CTO, Chief, Officer, etc. from the first three words and see if that helps
remove_titles = lambda x: " ".join([word for word in x.split() if word.lower() not in ["ceo", "cto", "chief", "officer", "cfo", "founder", "to", "in", "board"]])
df["first_three_words_no_verbs_no_s_no_titles"] = df["first_three_words_no_verbs_no_s"].apply(remove_titles)

In [7]:
company_names = df["first_three_words_no_verbs_no_s_no_titles"].value_counts().keys().to_list()

In [8]:
### check if the company names that are one word are present again in the company names that are two words or three words
one_word_company_names = [name for name in company_names if len(name.split()) == 1]
two_word_company_names = [name for name in company_names if len(name.split()) == 2]
three_word_company_names = [name for name in company_names if len(name.split()) == 3]

for name in one_word_company_names:
    if name in two_word_company_names:
        print(f"{name} is present in two word company names")
    if name in three_word_company_names:
        print(f"{name} is present in three word company names")

for name in two_word_company_names:
    if name in three_word_company_names:
        print(f"{name} is present in three word company names")

In [9]:
len(one_word_company_names)

0

In [10]:
df.head()

,session,headline,bar_ix,first_three_words,first_three_words_no_verbs,first_three_words_no_verbs_no_s,first_three_words_no_verbs_no_s_no_titles
0,0,Relvos Biosciences opens new office in Southea...,6,Relvos Biosciences opens,Relvos Biosciences opens,Relvos Biosciences,Relvos Biosciences
1,0,Orevex Renewables secures $500M contract with ...,12,Orevex Renewables secures,Orevex Renewables secures,Orevex Renewables,Orevex Renewables
2,0,Relvos Biosciences names new head of precision...,14,Relvos Biosciences names,Relvos Biosciences names,Relvos Biosciences,Relvos Biosciences
3,0,Calvis Sciences secures $650M contract with a ...,20,Calvis Sciences secures,Calvis Sciences secures,Calvis Sciences,Calvis Sciences
4,0,Yorvov Pharmaceuticals secures $180M contract ...,22,Yorvov Pharmaceuticals secures,Yorvov Pharmaceuticals secures,Yorvov Pharmaceuticals,Yorvov Pharmaceuticals


In [11]:
### For each company name, create which sessions and bars it is present in the training data
company_name_sessions = {}
for name in company_names:
    sessions = df[df["first_three_words_no_verbs_no_s_no_titles"] == name]["session"].unique().tolist()
    bars = df[df["first_three_words_no_verbs_no_s_no_titles"] == name]["bar_ix"].unique().tolist()
    company_name_sessions[name] = {"sessions": sessions, "bars": bars}

In [12]:
### check if it present at the beginning of the headline
company_name_sessions_beginning = {}
for name in company_names:
    sessions = df[df["headline"].str.startswith(name)]["session"].unique().tolist()
    bars = df[df["headline"].str.startswith(name)]["bar_ix"].unique().tolist()
    company_name_sessions_beginning[name] = {"sessions": sessions, "bars": bars}

In [13]:
for name in company_names:
    if len(company_name_sessions[name]['sessions']) != len(company_name_sessions_beginning[name]['sessions']):
        print(f"{name} is not present at the beginning of the headline in all sessions")
        print(f"Headline where it is not present at the beginning: {df[df['headline'].str.contains(name) & ~df['headline'].str.startswith(name)]['headline'].tolist()}")

In [14]:
company_names.sort()

In [15]:
df

,session,headline,bar_ix,first_three_words,first_three_words_no_verbs,first_three_words_no_verbs_no_s,first_three_words_no_verbs_no_s_no_titles
0,0,Relvos Biosciences opens new office in Southea...,6,Relvos Biosciences opens,Relvos Biosciences opens,Relvos Biosciences,Relvos Biosciences
1,0,Orevex Renewables secures $500M contract with ...,12,Orevex Renewables secures,Orevex Renewables secures,Orevex Renewables,Orevex Renewables
2,0,Relvos Biosciences names new head of precision...,14,Relvos Biosciences names,Relvos Biosciences names,Relvos Biosciences,Relvos Biosciences
3,0,Calvis Sciences secures $650M contract with a ...,20,Calvis Sciences secures,Calvis Sciences secures,Calvis Sciences,Calvis Sciences
4,0,Yorvov Pharmaceuticals secures $180M contract ...,22,Yorvov Pharmaceuticals secures,Yorvov Pharmaceuticals secures,Yorvov Pharmaceuticals,Yorvov Pharmaceuticals
...,...,...,...,...,...,...,...
17355,999,Yorval Partners begins scheduled maintenance o...,33,Yorval Partners begins,Yorval Partners begins,Yorval Partners,Yorval Partners
17356,999,Yorval Partners announces breakthrough in auto...,33,Yorval Partners announces,Yorval Partners announces,Yorval Partners,Yorval Partners
17357,999,Krevan Investments withdraws from Scandinavia ...,41,Krevan Investments withdraws,Krevan Investments withdraws,Krevan Investments,Krevan Investments
17358,999,Wrelal Financial secures $800M contract with a...,45,Wrelal Financial secures,Wrelal Financial secures,Wrelal Financial,Wrelal Financial


In [16]:
len(company_names)

80

In [17]:
# Ignore the first name and that is the company sector
sector = [" ".join(x.split()[1:]) for x in company_names]

In [23]:
df["sector"] = df["first_three_words_no_verbs_no_s_no_titles"].apply(lambda x: " ".join(x.split()[1:]))

In [24]:
df

,session,headline,bar_ix,first_three_words,first_three_words_no_verbs,first_three_words_no_verbs_no_s,first_three_words_no_verbs_no_s_no_titles,sector
0,0,Relvos Biosciences opens new office in Southea...,6,Relvos Biosciences opens,Relvos Biosciences opens,Relvos Biosciences,Relvos Biosciences,Biosciences
1,0,Orevex Renewables secures $500M contract with ...,12,Orevex Renewables secures,Orevex Renewables secures,Orevex Renewables,Orevex Renewables,Renewables
2,0,Relvos Biosciences names new head of precision...,14,Relvos Biosciences names,Relvos Biosciences names,Relvos Biosciences,Relvos Biosciences,Biosciences
3,0,Calvis Sciences secures $650M contract with a ...,20,Calvis Sciences secures,Calvis Sciences secures,Calvis Sciences,Calvis Sciences,Sciences
4,0,Yorvov Pharmaceuticals secures $180M contract ...,22,Yorvov Pharmaceuticals secures,Yorvov Pharmaceuticals secures,Yorvov Pharmaceuticals,Yorvov Pharmaceuticals,Pharmaceuticals
...,...,...,...,...,...,...,...,...
17355,999,Yorval Partners begins scheduled maintenance o...,33,Yorval Partners begins,Yorval Partners begins,Yorval Partners,Yorval Partners,Partners
17356,999,Yorval Partners announces breakthrough in auto...,33,Yorval Partners announces,Yorval Partners announces,Yorval Partners,Yorval Partners,Partners
17357,999,Krevan Investments withdraws from Scandinavia ...,41,Krevan Investments withdraws,Krevan Investments withdraws,Krevan Investments,Krevan Investments,Investments
17358,999,Wrelal Financial secures $800M contract with a...,45,Wrelal Financial secures,Wrelal Financial secures,Wrelal Financial,Wrelal Financial,Financial


In [19]:
#get unique sectors
unique_sectors = set(sector)

In [22]:
unique_sectors

{'Advisors',
 'Asset Group',
 'Biopharma',
 'Biosciences',
 'Biotech',
 'Brands',
 'Capital',
 'Commerce',
 'Computing',
 'Devices',
 'Diagnostics',
 'Energy',
 'Financial',
 'Fuels',
 'Genomics',
 'Goods',
 'Grid',
 'Holdings',
 'Industries',
 'Investments',
 'Labs',
 'Marketplace',
 'Microchips',
 'Networks',
 'Outlets',
 'Partners',
 'Pharmaceuticals',
 'Power',
 'Renewables',
 'Resources',
 'Retail Group',
 'Sciences',
 'Securities',
 'Software',
 'Solutions',
 'Stores',
 'Systems',
 'Technologies',
 'Therapeutics',
 'Trading Co'}

In [65]:
## Show the headlines with "Marketplace" as the first word
df[df["headline"].str.startswith("Marketplace")]["headline"].tolist()

[]

headline
Frelol Software misses quarterly revenue estimates by 18%                                5
Dralol Computing completes planned facility upgrade in Latin America                     4
Zrovov Advisors secures $180M contract with a global retailer                            4
Orevar Marketplace reports record quarterly revenue, up 5% year-over-year                4
Creven Securities secures $320M contract with a top-tier research institute              3
                                                                                        ..
Yorval Partners begins scheduled maintenance of enterprise software systems              1
Yorval Partners announces breakthrough in automated logistics                            1
Krevan Investments withdraws from Scandinavia market citing unfavorable conditions       1
Wrelal Financial secures $800M contract with a global retailer                           1
Zelvix Therapeutics recalls products in digital payments line due to quality conc

In [66]:
company_name_sessions['Zelval Energy']['bars']

[30,
 31,
 15,
 38,
 13,
 2,
 25,
 40,
 35,
 23,
 19,
 49,
 7,
 12,
 42,
 43,
 0,
 18,
 24,
 27,
 29,
 32,
 1,
 5,
 14,
 39,
 8,
 21,
 33,
 46,
 4,
 37,
 9,
 36,
 34,
 48,
 6,
 26,
 28,
 17,
 3,
 20,
 10,
 44,
 11,
 22,
 47,
 16,
 45,
 41]